In [12]:
from glove_word_embeddings import pre,val,cat,mod
from collections import Counter, defaultdict
from nltk.corpus import wordnet as wn
import numpy as np, nltk; nltk.download("wordnet")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/daweiwang/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Preprocessing

In [2]:
# Individual cleaning modules
print(pre.return_lowercase("  Cat! "))
print(pre.strip_marks("  Cat! "))
print(pre.strip_stopword("The Cat"))
print(pre.strip_space("jar of jam"))
print(pre.remove_stopwords("jar of jam"))
print(pre.space_check("cat"))
print(pre.space_check("jar of jam"))

  cat! 
  Cat  
Cat
jarofjam
['jar', 'jam']
True
False


In [3]:
# Full cleaning module
print(pre.clean_word("  The Cat! "))
print(pre.clean_word("jar of jam"))
print(pre.clean_word("jar of jam", strip_space_bool=False))
print(pre.clean_word("the", strip_stopword_bool=False))

cat
jarjam
jar jam
the


In [4]:
# WordNet lemmatization module
print(pre.lemmatize("dogs"))
print(pre.lemmatize("running", pos="v"))

dog
run


### Validation

In [5]:
# WordNet validations
print(val.word("telescope"))
print(val.word("jar of jam"))
print(val.noun("telescope"))
print(val.noun("quickly"))

True
False
True
False


In [6]:
# Olson et al., 2021's wordlist
m = mod.load("glove-olson-validated")
v = m.vocab_set()
print(val.vocab("telescope", v))
print(val.vocab("ice cream", v))

True
False


### Categorization (Bucket)

In [7]:
# Ordinary common word?
print(cat.check_common("apple"))
print(cat.check_common("nike"))
print(cat.check_common("aaron"))

True
False
False


In [8]:
# Which curated buckets does a word belong to?
print(cat.check_bucket("dog"))
print(cat.check_bucket("paris", check_common=False))
print(cat.check_bucket("apple", check_common=True))
print(cat.check_bucket("apple", check_common=False))

{'animals'}
{'places'}
{'fruits', 'plants'}
{'fruits', 'brands', 'plants'}


In [9]:
# Count across a list of words
words = ['test',                    # 1 one-word subjects
         'dog','cat',               # 2 one-word animals
         'apple',                   # Brand and object
         'chair','table','desk',    # 3 one-word objects         
         'sun glasses',             # 2 compound
         'wr0ng','w0rd']            # 2 wrong words

print(cat.count_bucket(words, 3))
print(cat.count_bucket(words, 4))
print(cat.count_bucket(words, name="environment", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="animals", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="brands", number_of_words=1, check_common=False))
print(cat.count_bucket(words, name="places", number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="names",  number_of_words=1, check_common=True))
print(cat.count_bucket(words, name="brands", number_of_words=1, check_common=True))

True
False
True
True
True
False
False
False


### Categorization (WordNet)

In [15]:
depth_counts = Counter()
unique_names = defaultdict(set)

for synset in wn.all_synsets(pos="n"):
    d = synset.min_depth()               # shortest path length to entity
    depth_counts[d] += 1
    name = synset.name().split(".")[0]   # e.g. "dog" instead of "dog.n.01"
    unique_names[d].add(name)

print("Noun categories at each level (min_depth from entity):\n")
print(f"{'Level':>6}  {'Synsets':>8}  {'Unique names':>12}")
print("-" * 32)
for level in sorted(depth_counts):
    print(f"{level:6d}  {depth_counts[level]:8d}  {len(unique_names[level]):12d}")

print(f"Total noun synsets : {sum(depth_counts.values())}")
print(f"Maximum depth      : {max(depth_counts)}")

Noun categories at each level (min_depth from entity):

 Level   Synsets  Unique names
--------------------------------
     0         1             1
     1         3             3
     2        22            22
     3       228           216
     4      2020          1918
     5      6249          5790
     6     12267         11129
     7     18936         17636
     8     14155         13412
     9     11042         10539
    10      7207          6993
    11      4267          4174
    12      2505          2453
    13      1383          1366
    14       846           838
    15       449           443
    16       341           338
    17       164           158
    18        30            30
Total noun synsets : 82115
Maximum depth      : 18


In [17]:
def synset_depth(ss):
    """Minimum depth from the root (entity.n.01)."""
    return min(len(p) for p in ss.hypernym_paths())

# ----- 1. Depth distribution of all noun synsets -----
depths = []
for ss in wn.all_synsets("n"):
    depths.append(synset_depth(ss))

depths = np.array(depths)
print("Total noun synsets:", len(depths))
print("Max depth         :", depths.max())

print("Synsets at each exact depth:")
cnt = Counter(depths)
for d in range(0, int(depths.max()) + 1):
    print(f"  level {d:2d}: {cnt[d]:6d}")

print("Cumulative – fraction that reach at least depth k:")
for k in range(0, 12):
    frac = (depths >= k).mean()
    print(f"  ≥ {k:2d}: {frac:.3f}  ({(depths >= k).sum()} synsets)")

# ----- 2. Is every branch able to grow to level 8? -----
# (i.e. does every synset that exists at level 7 have at least one hyponym path that continues?)
reach_8 = (depths >= 8).sum()
print(f"Synsets that reach depth 8 or deeper: {reach_8}")
print(f"Fraction of all noun synsets that reach ≥ 8: {(depths >= 8).mean():.3f}")

# ----- 3. Density comparison -----
print("Density (number of synsets) around the peak:")
for d in range(5, 11):
    print(f"  level {d}: {cnt[d]}")

Total noun synsets: 82115
Max depth         : 19
Synsets at each exact depth:
  level  0:      0
  level  1:      1
  level  2:      3
  level  3:     22
  level  4:    228
  level  5:   2020
  level  6:   6249
  level  7:  12267
  level  8:  18936
  level  9:  14155
  level 10:  11042
  level 11:   7207
  level 12:   4267
  level 13:   2505
  level 14:   1383
  level 15:    846
  level 16:    449
  level 17:    341
  level 18:    164
  level 19:     30
Cumulative – fraction that reach at least depth k:
  ≥  0: 1.000  (82115 synsets)
  ≥  1: 1.000  (82115 synsets)
  ≥  2: 1.000  (82114 synsets)
  ≥  3: 1.000  (82111 synsets)
  ≥  4: 1.000  (82089 synsets)
  ≥  5: 0.997  (81861 synsets)
  ≥  6: 0.972  (79841 synsets)
  ≥  7: 0.896  (73592 synsets)
  ≥  8: 0.747  (61325 synsets)
  ≥  9: 0.516  (42389 synsets)
  ≥ 10: 0.344  (28234 synsets)
  ≥ 11: 0.209  (17192 synsets)
Synsets that reach depth 8 or deeper: 61325
Fraction of all noun synsets that reach ≥ 8: 0.747
Density (number of synse

In [25]:
# Full ladder (specific → general)
print('fullest primary:',cat.category_chain("badger", path='full', sense='primary'))
print('shortest primary:',cat.category_chain("badger", path='short', sense='primary'))
print('longest primary:',cat.category_chain("badger", path='long', sense='primary'))
print('fullest union:',cat.category_chain("badger", path='full', sense='union'))
print('shortest union:',cat.category_chain("badger", path='short', sense='union'))
print('longest union:',cat.category_chain("badger", path='long', sense='union'))

# Single rung (0 = most specific)
print(cat.category_by_level("dog", level=0))
print(cat.category_by_level("dog", level=4))

# Do two words share the same category at a given level?
print(cat.check_same_category("dog", "wolf"))
print(cat.check_same_category("dog", "wolf", level=1))

# Most specific shared category name / its level on word1’s chain
print(cat.category_shared_name("dog", "wolf"))
print(cat.category_shared_level("dog", "wolf"))

fullest primary: ['entity', 'physical_entity', 'causal_agent', 'object', 'whole', 'living_thing', 'organism', 'person', 'inhabitant', 'american']
shortest primary: ['entity', 'physical_entity', 'causal_agent', 'person', 'inhabitant', 'american']
longest primary: ['entity', 'physical_entity', 'object', 'whole', 'living_thing', 'organism', 'person', 'inhabitant', 'american']
fullest union: ['entity', 'physical_entity', 'causal_agent', 'object', 'whole', 'living_thing', 'organism', 'animal', 'person', 'chordate', 'inhabitant', 'american', 'vertebrate', 'mammal', 'placental', 'carnivore', 'musteline_mammal']
shortest union: ['entity', 'physical_entity', 'causal_agent', 'person', 'inhabitant', 'american']
longest union: ['entity', 'physical_entity', 'object', 'whole', 'living_thing', 'organism', 'animal', 'chordate', 'vertebrate', 'mammal', 'placental', 'carnivore', 'musteline_mammal']
entity
living_thing
True
True
entity
0


### Embedding

In [26]:
m = mod.load("glove-6b-300d")     # downloads on first use, caches locally
m.embed_exact("cat")              # exact match only → np.ndarray or None
m.vocab_set()                     # → set of all words in the model

# Multi-word phrases
m.embed_phrase("jar of jam")      # tries exact / underscore / hyphen variants,
                                  # otherwise averages the non-stopword parts

array([-5.39150089e-02, -1.67890012e-01, -3.39424998e-01, -4.79775012e-01,
        5.34095049e-01, -4.70174998e-01,  1.87825009e-01,  2.56150067e-02,
       -1.66271001e-01,  3.60450000e-02, -2.29700506e-01,  2.06964999e-01,
       -2.28724003e-01,  4.25989985e-01,  2.42854983e-01,  4.13580000e-01,
       -3.78210008e-01, -1.14800036e-02, -2.71800086e-02,  3.84159982e-01,
        4.52730000e-01,  4.20230031e-02, -4.64850068e-02,  1.74560502e-01,
       -9.19149965e-02, -3.51429999e-01, -5.49199954e-02, -4.59495008e-01,
        4.21205014e-01, -4.38349992e-02, -3.16745013e-01,  1.27987504e-01,
        1.48792997e-01, -1.76585004e-01, -1.96496993e-01,  4.95510012e-01,
       -2.10300088e-02,  1.20100006e-01, -2.33635992e-01, -2.11590007e-01,
       -2.06441998e-01, -1.13584995e-01, -1.25944003e-01,  6.84195042e-01,
       -9.71650034e-02,  2.07500160e-03,  8.67950022e-02, -2.11193502e-01,
        1.25755504e-01,  4.25349995e-02,  3.96725506e-01,  2.53185004e-01,
       -2.25359991e-01, -